<a href="https://colab.research.google.com/github/data-geass/machine-learning-practice/blob/main/credit_scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
# загрузим данные
! pip install wldhx.yadisk-direct
! curl -L $(yadisk-direct https://disk.yandex.com/d/sknuSa3xoNBsDw) -o bank-issues-data.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3473k  100 3473k    0     0  1495k      0  0:00:02  0:00:02 --:--:-- 3920k


In [27]:
! unzip -o -qq bank-issues-data.zip

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [29]:
train_data = pd.read_csv('./bank-issues-data/train.csv')
test_data = pd.read_csv('./bank-issues-data/test.csv')

In [30]:
train_data

,client_id,gender,age,marital_status,job_position,credit_sum,credit_month,tariff_id,score_shk,education,living_region,monthly_income,credit_count,overdue_credit_count,open_account_flg
0,1,M,25,UNM,SPC,26389.0,10,1.32,0.584105,SCH,ОБЛ КУРСКАЯ,35000.0,2.0,0.0,1
1,2,F,37,MAR,SPC,19588.0,12,1.43,0.718935,SCH,РЕСПУБЛИКА ТАТАРСТАН,15000.0,0.0,0.0,1
2,3,F,28,UNM,SPC,53669.0,18,1.10,0.586015,GRD,МОСКВА Г,70000.0,4.0,0.0,1
3,4,M,34,MAR,SPC,26349.0,10,1.43,0.655703,SCH,СВЕРДЛОВСКАЯ ОБЛАСТЬ,42500.0,4.0,0.0,0
4,5,F,43,MAR,UMN,11589.0,10,1.10,0.271893,GRD,РЯЗАНСКАЯ ОБЛАСТЬ,20000.0,3.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119513,119514,F,29,UNM,SPC,21490.0,12,1.40,0.551201,GRD,САНКТ-ПЕТЕРБУРГ Г,65000.0,2.0,0.0,0
119514,119515,M,40,MAR,SPC,45028.0,10,1.60,0.698376,SCH,ТАТАРСТАН РЕСП,50000.0,NaN,NaN,0
119515,119516,F,22,UNM,SPC,20990.0,12,1.90,0.570818,GRD,ОБЛ КУРГАНСКАЯ,20000.0,1.0,0.0,1
119516,119517,F,34,UNM,SPC,17108.0,10,1.10,0.442541,GRD,КРАСНОДАРСКИЙ КРАЙ,17000.0,0.0,0.0,0


In [31]:
y_train = train_data['open_account_flg']
train_data = train_data.drop(columns='open_account_flg')

In [32]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119518 entries, 0 to 119517
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   client_id             119518 non-null  int64  
 1   gender                119518 non-null  object 
 2   age                   119518 non-null  int64  
 3   marital_status        119518 non-null  object 
 4   job_position          119518 non-null  object 
 5   credit_sum            119518 non-null  float64
 6   credit_month          119518 non-null  int64  
 7   tariff_id             119518 non-null  float64
 8   score_shk             119518 non-null  float64
 9   education             119518 non-null  object 
 10  living_region         119385 non-null  object 
 11  monthly_income        119518 non-null  float64
 12  credit_count          113032 non-null  float64
 13  overdue_credit_count  113032 non-null  float64
dtypes: float64(6), int64(3), object(5)
memory usage: 12.

важно отделять категориальные признаки от числовых, просто float недостаточно

In [33]:
train_data.isna().any()

,0
client_id,False
gender,False
age,False
marital_status,False
job_position,False
credit_sum,False
credit_month,False
tariff_id,False
score_shk,False
education,False


In [34]:
test_data.isna().any()

,0
client_id,False
gender,False
age,False
marital_status,False
job_position,False
credit_sum,False
credit_month,False
tariff_id,False
score_shk,False
education,False


Видим интересную вещь, что в monthly_income есть пропуски! в отличии от train_data. При этом, если просто удалить такие строки мы не можем (например, в том же Kaggle надо выдать ответ на все строки из тестовой таблицы) то их надо будет заполнить.

Чтобы заполнить эти пропуски, можно использовать медиану (лучше ее нежели ср. значение, т.к. она менее восприимчива к аномалиям/выбрасам) или методы из sklearn

In [35]:
train_data['credit_count'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 119518 entries, 0 to 119517
Series name: credit_count
Non-Null Count   Dtype  
--------------   -----  
113032 non-null  float64
dtypes: float64(1)
memory usage: 933.9 KB


In [36]:
119518 - 113032

6486

In [37]:
# давайте попробуем в ручную заполнить медианой
# median_value_credit_count = np.median(train_data['credit_count'].dropna())
# train_data['credit_count'].fillna(median_value_credit_count)

In [38]:
# можно и не в ручную))
from sklearn.impute import SimpleImputer

simple_imputer = SimpleImputer(missing_values=np.nan, strategy='median')
simple_imputer.fit(train_data[['credit_count']])
train_data['credit_count'] = simple_imputer.transform(train_data[['credit_count']])

In [39]:
# дальше для заполнения можно использовать knn метод
# он принимает только числовые столбцы, тк ищется
# евклидово расстояние между векторами
train_data['job_position'].dtype

dtype('O')

In [40]:
# видим, что у object тайп 'O' -> вот их и надо исключить
# создается список типо ['age', 'credit_sum', 'credit_count'] ->
numeric_columns = [column for column in train_data.columns if train_data[column].dtype != 'O']
numeric_columns


['client_id',
 'age',
 'credit_sum',
 'credit_month',
 'tariff_id',
 'score_shk',
 'monthly_income',
 'credit_count',
 'overdue_credit_count']

In [41]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=3)
knn_imputer.fit(train_data[numeric_columns])
transformed_overdue_credit_count = knn_imputer.transform(train_data[numeric_columns])[:, 8]
train_data['overdue_credit_count'] = transformed_overdue_credit_count

In [42]:
train_data['overdue_credit_count'].isna().any()

np.False_

# Обработка категориальных признаков

In [43]:
categorical_columns = [column for column in train_data.columns if train_data[column].dtype == 'O']
categorical_columns

['gender', 'marital_status', 'job_position', 'education', 'living_region']

In [44]:
# tariff_id - с ним нельзя считать математику
# поэтому это тоже категориальный
categorical_columns += ['tariff_id']
categorical_columns.remove('living_region')
categorical_columns

['gender', 'marital_status', 'job_position', 'education', 'tariff_id']

In [45]:
np.unique(train_data['marital_status'])

array(['CIV', 'DIV', 'MAR', 'UNM', 'WID'], dtype=object)

In [46]:
# чтобы не попасть в ловушку ранжирования
# мы каждый из этих значений ['CIV', 'DIV', 'MAR', 'UNM', 'WID']
# сделаем отдельным столбцом/признаком
# это называется One-hot кодирование
from sklearn.preprocessing import OneHotEncoder

# drop='if_binary' - если столбец имеет и так два значения то отдельный столбец не нужен
ohe = OneHotEncoder(drop='if_binary', sparse_output=False)
ohe.fit(train_data[categorical_columns])
new_category_columns = ohe.transform(train_data[categorical_columns])

п.с. название «One-Hot» пришло из цифровой электроники. Если провод находится под напряжением (логическая единица 1), его называют «горячим».

In [47]:
new_category_columns

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]])

In [48]:
ohe.get_feature_names_out()

array(['gender_M', 'marital_status_CIV', 'marital_status_DIV',
       'marital_status_MAR', 'marital_status_UNM', 'marital_status_WID',
       'job_position_ATP', 'job_position_BIS', 'job_position_BIU',
       'job_position_DIR', 'job_position_HSK', 'job_position_INP',
       'job_position_NOR', 'job_position_PNA', 'job_position_PNI',
       'job_position_PNS', 'job_position_PNV', 'job_position_SPC',
       'job_position_UMN', 'job_position_WOI', 'job_position_WRK',
       'job_position_WRP', 'education_ACD', 'education_GRD',
       'education_PGR', 'education_SCH', 'education_UGR', 'tariff_id_1.0',
       'tariff_id_1.1', 'tariff_id_1.16', 'tariff_id_1.17',
       'tariff_id_1.18', 'tariff_id_1.19', 'tariff_id_1.2',
       'tariff_id_1.21', 'tariff_id_1.22', 'tariff_id_1.23',
       'tariff_id_1.24', 'tariff_id_1.25', 'tariff_id_1.26',
       'tariff_id_1.27', 'tariff_id_1.28', 'tariff_id_1.29',
       'tariff_id_1.3', 'tariff_id_1.32', 'tariff_id_1.4',
       'tariff_id_1.41', 'tarif

In [49]:
new_train_columns = pd.DataFrame(new_category_columns, columns=ohe.get_feature_names_out())
train_data = train_data.drop(columns=categorical_columns)
train_data = pd.concat([train_data, new_train_columns], axis=1)
train_data

,client_id,age,credit_sum,credit_month,score_shk,living_region,monthly_income,credit_count,overdue_credit_count,gender_M,...,tariff_id_1.48,tariff_id_1.5,tariff_id_1.52,tariff_id_1.56,tariff_id_1.6,tariff_id_1.7,tariff_id_1.9,tariff_id_1.91,tariff_id_1.94,tariff_id_1.96
0,1,25,26389.0,10,0.584105,ОБЛ КУРСКАЯ,35000.0,2.0,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,37,19588.0,12,0.718935,РЕСПУБЛИКА ТАТАРСТАН,15000.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,28,53669.0,18,0.586015,МОСКВА Г,70000.0,4.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,34,26349.0,10,0.655703,СВЕРДЛОВСКАЯ ОБЛАСТЬ,42500.0,4.0,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,43,11589.0,10,0.271893,РЯЗАНСКАЯ ОБЛАСТЬ,20000.0,3.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119513,119514,29,21490.0,12,0.551201,САНКТ-ПЕТЕРБУРГ Г,65000.0,2.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
119514,119515,40,45028.0,10,0.698376,ТАТАРСТАН РЕСП,50000.0,2.0,0.333333,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
119515,119516,22,20990.0,12,0.570818,ОБЛ КУРГАНСКАЯ,20000.0,1.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
119516,119517,34,17108.0,10,0.442541,КРАСНОДАРСКИЙ КРАЙ,17000.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Обработка living_region

In [53]:
np.unique(train_data['living_region'].dropna())

array(['74', '98', 'АДЫГЕЯ РЕСП', 'АЛТАЙСКИЙ', 'АЛТАЙСКИЙ КРАЙ',
       'АМУРСКАЯ ОБЛ', 'АМУРСКАЯ ОБЛАСТЬ', 'АО НЕНЕЦКИЙ',
       'АО ХАНТЫ-МАНСИЙСКИЙ АВТОНОМНЫЙ ОКРУГ - Ю', 'АО ЯМАЛО-НЕНЕЦКИЙ',
       'АОБЛ ЕВРЕЙСКАЯ', 'АРХАНГЕЛЬСКАЯ', 'АРХАНГЕЛЬСКАЯ ОБЛ',
       'АРХАНГЕЛЬСКАЯ ОБЛАСТЬ', 'АСТРАХАНСКАЯ', 'АСТРАХАНСКАЯ ОБЛ',
       'АСТРАХАНСКАЯ ОБЛАСТЬ', 'БАШКОРТОСТАН', 'БАШКОРТОСТАН РЕСП',
       'БЕЛГОРОДСКАЯ ОБЛ', 'БЕЛГОРОДСКАЯ ОБЛАСТЬ', 'БРЯНСКАЯ ОБЛ',
       'БРЯНСКАЯ ОБЛАСТЬ', 'БРЯНСКИЙ', 'БУРЯТИЯ', 'БУРЯТИЯ РЕСП',
       'ВЛАДИМИРСКАЯ ОБЛ', 'ВЛАДИМИРСКАЯ ОБЛАСТЬ', 'ВОЛГОГРАДСКАЯ ОБЛ',
       'ВОЛГОГРАДСКАЯ ОБЛАСТЬ', 'ВОЛОГОДСКАЯ', 'ВОЛОГОДСКАЯ ОБЛ',
       'ВОЛОГОДСКАЯ ОБЛ.', 'ВОЛОГОДСКАЯ ОБЛАСТЬ', 'ВОРОНЕЖСКАЯ ОБЛ',
       'ВОРОНЕЖСКАЯ ОБЛАСТЬ', 'Г МОСКВА', 'Г. МОСКВА',
       'Г. САНКТ-ПЕТЕРБУРГ', 'Г.МОСКВА', 'ГОРЬКОВСКАЯ ОБЛ',
       'ГУСЬ-ХРУСТАЛЬНЫЙ Р-Н', 'ДАГЕСТАН РЕСП', 'ЕВРЕЙСКАЯ АВТОНОМНАЯ',
       'ЕВРЕЙСКАЯ АОБЛ', 'ЗАБАЙКАЛЬСКИЙ КРАЙ', 'ИВАНОВСКАЯ ОБЛ',
       'ИВАНОВ